# 레인 C-1 · 제외 7건 실행

레인 B(월간 자동 실행)가 **용량·GPU 때문에 등록하지 않은 노트북 7편**을 돌린다.
이 7편은 **한 번도 자동으로 돈 적이 없다.**

> **무료 등급 Colab 을 전제로 만들었다.** 한 번에 다 돌지 못해도 되게 짜여 있다.

## 어떻게 쓰는가

1. **런타임 → 런타임 유형 변경 → T4 GPU** 로 바꾼다. (이게 없으면 VII-2 는 못 돈다)
2. **셀 1(준비)을 먼저 실행한다.** 저장소를 받고 대역을 켠다. 30초쯤 걸린다.
3. 그다음은 **부분마다 셀이 하나씩**이다. 위에서부터 차례로 실행한다.
4. 부분이 끝날 때마다 **그때까지의 결과표가 찍힌다.**

## 입력 파일은 알아서 가져온다

VIII-2·VIII-3 에 필요한 파일 셋을 **부분 B 가 드라이브를 붙여 스스로 찾아온다.**
권한 창이 한 번 뜨면 허용하면 된다. `/content/` 에 이미 있으면 드라이브를 붙이지 않는다.
손으로 올릴 필요가 없다.

## ⚠ 세션이 끊기면

무료 등급은 GPU 가 회수되거나 세션이 끊길 수 있다. **끊겨도 앞의 결과는 이미 화면에 남아 있다.**
이어서 돌 때는 **셀 1(준비)만 다시 실행한 뒤 못 끝낸 부분 셀부터** 누르면 된다.
부분끼리는 서로 의존하지 않는다.

## 순서를 이렇게 둔 이유

| 부분 | 노트북 | 왜 이 자리인가 |
|---|---|---|
| **A** | VII-2 Diffusion | **유일하게 T4 + fp16 이 진짜로 필요하다.** GPU 를 받은 그 세션에서 가장 먼저 해치운다 |
| **B** | VII-3① Music · VIII-2 Captioning · VIII-3 Subtitle | 중간 무게. 각각 독립적으로 끝난다 |
| **C** | VI-4 · VI-5 · VI-9번역 | **가장 무겁고 가장 오래.** 끊길 확률이 높아 맨 뒤에 둔다 |

**총 내려받기 약 11GB.** 부분 C 만 4GB 가 넘는다.

## 마지막 셀

**판정표 두 칸**을 찍는다. 위는 코드가 판정한 것, 아래는 저자가 눈으로 볼 것이다.
**두 표를 통째로 복사해 지침서에 붙여넣는다.**

---

In [ ]:
# ── 셀 1 · 준비 ───────────────────────────────────────────────────────────────
# 저장소를 받고, 노트북을 '있는 그대로' 실행할 장치를 켠다.
#
# ⚠ 레인 B 와 다른 점이 둘 있다. 그게 이 레인의 존재 이유다.
#   ① `!pip` 줄을 지우지 않는다. **고정 판번호 그대로 처음부터 돌린다.**
#   ② matplotlib 을 Agg 로 바꾸지 않는다. **그림이 실제로 나오는지 봐야 하기 때문이다.**

import subprocess, tarfile, glob, os, sys, json, time, io, contextlib, traceback

t0 = time.time()
subprocess.run(["curl", "-sL", "-o", "/content/repo.tgz",
                "https://codeload.github.com/MLFundamentals/making-ai/tar.gz/refs/heads/main"],
               check=True)
with tarfile.open("/content/repo.tgz") as t:
    t.extractall("/content/repo")
ROOT = glob.glob("/content/repo/making-ai-*")[0]
NB_DIR = os.path.join(ROOT, "notebooks")
print(f"저장소 받음 ({time.time()-t0:.0f}초) · 노트북 {len(glob.glob(NB_DIR+'/*.ipynb'))}편")

# ── input() 대역 ─────────────────────────────────────────────────────────────
# 답은 **노트북 저장 출력에서 그대로 꺼냈다.** 저자가 실제로 넣었던 값이라 책과 대조된다.
# 지어낸 값을 넣으면 '돌긴 했는데 책과 다른 것을 확인한' 상태가 된다.

_ANSWERS = []
_ASKED = []

class _TooManyInputs(RuntimeError):
    pass

def _fake_input(prompt=""):
    if prompt:
        print(prompt, end="")
    if not _ANSWERS:
        raise _TooManyInputs(
            "준비한 답보다 input() 호출이 많다. 노트북이 바뀌었을 수 있다.")
    a = _ANSWERS.pop(0)
    _ASKED.append(a)
    print(a)
    return a

# ── gradio 대역 ──────────────────────────────────────────────────────────────
# .launch() 는 화면을 띄우려 하므로 막는다. 대신 만들어진 Interface 를 붙잡아 두고,
# 그 `fn` 을 **우리가 직접 부른다**. VIII-2·VIII-3 이 레인 B 에서 제외된 이유가
# '입력이 독자가 올리는 파일이라 대역이 꺼낼 값이 없다'였는데,
# 레인 C 에는 그 파일이 실제로 있다.

_INTERFACES = []
_GR_HOOKED = [False]

def _install_gradio_hook():
    if _GR_HOOKED[0]:
        return True
    _GR_HOOKED[0] = True          # 먼저 세운다 — 아래 import 가 이 함수를 다시 부른다
    try:
        import gradio as gr
    except ImportError:
        _GR_HOOKED[0] = False
        return False
    _orig_init = gr.Interface.__init__
    def _init(self, *a, **kw):
        _orig_init(self, *a, **kw)
        _INTERFACES.append(self)
    gr.Interface.__init__ = _init
    # Interface 는 Blocks 를 상속한다. Blocks 쪽을 막으면 둘 다 막힌다.
    target = getattr(gr, "Blocks", gr.Interface)
    target.launch = lambda self, *a, **kw: print("   [대역] .launch() 는 건너뛴다")
    print("   [대역] gradio 붙잡음")
    return True


def _make_builtins():
    """gradio 가 import 되는 **그 순간** 대역을 건다.

    ⚠ VIII-2·VIII-3 은 설치·import·Interface 생성·.launch() 가 **전부 셀 0 안**에 있다.
      셀이 끝난 뒤에 대역을 걸면 이미 늦어서 진짜 서버가 뜬다.
      그래서 셀 밖이 아니라 `import` 자체에 갈고리를 건다.
    """
    import builtins
    b = dict(vars(builtins))
    _real = builtins.__import__
    def _imp(name, *a, **kw):
        m = _real(name, *a, **kw)
        if name.split(".")[0] == "gradio":
            _install_gradio_hook()
        return m
    b["__import__"] = _imp
    return b

# ── 셀 실행기 ────────────────────────────────────────────────────────────────
# 노트북을 **메모리로만** 읽어 셀을 하나씩 돌린다. 고칠 파일이 없으니 드리프트가 없다.
# 셀별 소요 시간을 남긴다 — '독자의 실제 소요 시간'이 이 레인의 목적 하나다.

class ShellError(RuntimeError):
    pass


def _shell(cmd):
    """`!cmd` 대신. 종료 코드가 0이 아니면 **그 자리에서 세운다.**

    조용한 실패가 이 프로젝트의 유일한 사고 유형이다. pip 가 실패했는데
    다음 셀의 ImportError 로 알게 되면 진짜 원인이 화면 밖으로 밀려난다.
    """
    ip = get_ipython()
    ip.system(cmd)
    rc = ip.user_ns.get("_exit_code", 0)
    if rc:
        raise ShellError(f"셸 종료 코드 {rc}: {cmd}")


RESULTS = {}     # 절 → dict(ok, secs, cells=[(i, 초, 상태)], err)

def run_notebook(section, filename, answers=None, only_cells=None,
                 install=True, globals_out=None):
    """노트북 한 편을 셀 순서대로 실행한다. 실행에 쓴 전역 namespace 를 돌려준다."""
    global _ANSWERS
    _ANSWERS = list(answers or [])
    _ASKED.clear()
    _INTERFACES.clear()

    path = os.path.join(NB_DIR, filename)
    nb = json.load(open(path, encoding="utf-8"))
    cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

    g = {"__name__": "__main__", "input": _fake_input, "get_ipython": get_ipython,
         "_shell": _shell, "__builtins__": _make_builtins()}
    timings, err = [], None
    print(f"\n{'='*70}\n▶ {section}  {filename}\n{'='*70}")

    whole = time.time()
    for i, c in enumerate(cells):
        if only_cells is not None and i not in only_cells:
            print(f"  [셀 {i}] 건너뜀 (only_cells 지정)")
            continue
        src = "".join(c["source"])
        # 셸 줄은 IPython 에게 넘긴다 — !pip 도 !wget 도 **진짜로** 실행한다.
        # `!cmd` 는 IPython 이 get_ipython().system(cmd) 로 바꾸는 것과 같게 맞춘다.
        # sx 를 쓰면 출력이 화면에 안 나오고 pip 오류가 조용히 묻힌다.
        lines = []
        for ln in src.splitlines():
            st = ln.strip()
            if st.startswith("!"):
                if not install and "pip" in st:
                    print(f"  [셀 {i}] 설치 줄 건너뜀: {st}")
                    continue
                lines.append(f"_shell({st[1:]!r})")
            elif st.startswith("%"):
                head, _, rest = st[1:].partition(" ")
                lines.append(f"get_ipython().run_line_magic({head!r}, {rest!r})")
            else:
                lines.append(ln)
        code = "\n".join(lines)

        t = time.time()
        try:
            exec(compile(code, f"<{filename} cell {i}>", "exec"), g)
            dt = time.time() - t
            timings.append((i, round(dt, 1), "OK"))
            print(f"  [셀 {i}] OK ({dt:.0f}초)")
        except Exception as e:
            dt = time.time() - t
            timings.append((i, round(dt, 1), type(e).__name__))
            err = traceback.format_exc()
            print(f"  [셀 {i}] 실패 ({dt:.0f}초) — {type(e).__name__}: {e}")
            print(err[-1500:])
            break

    total = round(time.time() - whole, 1)
    leftover = len(_ANSWERS)
    if leftover and err is None:
        err = f"쓰이지 않은 input() 답이 {leftover}개 남았다 — 노트북이 바뀌었을 수 있다"
        print(f"  ⚠ {err}")

    RESULTS[section] = dict(file=filename, ok=(err is None), secs=total,
                            cells=timings, err=(err or "")[-800:])
    print(f"  → {'완주' if err is None else '실패'} · 합계 {total:.0f}초")
    if globals_out is not None:
        globals_out.update(g)
    return g

# ── 셸 대역 확인 ─────────────────────────────────────────────────────────────
# Colab 의 IPython 을 그대로 쓴다. get_ipython() 이 없으면 !pip 가 안 돌아간다.
try:
    get_ipython
    print("IPython 있음 — 셸 줄(!pip · !wget)이 실제로 실행된다")
except NameError:
    raise SystemExit("이 노트북은 Colab(IPython)에서 돌려야 한다")

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 ⚠ VII-2 는 못 돈다'}")
print("\n준비 끝. 아래 부분 셀을 차례로 실행할 것.")

In [ ]:
# ── 셀 2 · 부분 A · VII-2 Diffusion ──────────────────────────────────────────
# **T4 + fp16 이 진짜로 필요한 유일한 노트북.** GPU 를 받은 지금 가장 먼저 돌린다.
# 걸림돌: 라이선스 게이트(stabilityai/sd-turbo). 막히면 그 자리에서 오류가 난다.
#
# ⚠ 이 노트북은 씨앗이 없다(seed=None → random). 매번 다른 그림이 나오는 것이 정상이다.
#   확인할 것은 '같은 그림'이 아니라 '프롬프트와 맞는 그림'이다.

g = run_notebook("VII-2", "Generative AI_Diffusion.ipynb")

print("\n── 눈으로 볼 것 ──")
img = g.get("image")
if img is not None:
    print(f"  이미지 크기: {img.size} · 모드: {img.mode}")
    print(f"  프롬프트: {g.get('user_prompt')}")
    print("  □ 위에 뜬 그림이 '우주복 입은 귀여운 개'인가?")
    print("  □ 원고의 예시 그림과 분위기가 비슷한가? (같은 그림일 필요는 없다)")
else:
    print("  ⚠ image 변수가 없다 — 셀 1이 끝까지 안 돌았다")

print(f"\n장치: {g.get('device')} · dtype: {g.get('dtype')}")
if g.get("device") != "cuda":
    print("  🔴 CPU 로 돌았다. 런타임 유형을 T4 GPU 로 바꾸고 다시 실행할 것.")

In [ ]:
# ── 셀 3 · 부분 B · VII-3① Music · VIII-2 Captioning · VIII-3 Subtitle ──────
# 중간 무게 셋. 각각 독립이라 하나가 실패해도 다음이 돈다.

# ---- VII-3① Music : MusicGen 2.36GB ----------------------------------------
# 마지막 줄이 Audio(...) 라 셀의 '값'으로만 남고 변수에 안 담긴다 → out 으로 확인한다.
g_music = run_notebook("VII-3①", "Generative AI_Music.ipynb")
out = g_music.get("out")
if out is not None:
    import numpy as np
    a = np.array(out["audio"]).squeeze()
    sr = out.get("sampling_rate", 32000)
    print(f"  오디오 {a.shape} · {sr}Hz · 약 {a.size/sr:.1f}초")
    from IPython.display import Audio, display
    display(Audio(a, rate=sr))
    print("  □ 위 재생 버튼을 눌러 소리가 나는가?")
    print("  □ 프롬프트(밝은 신스의 K-pop)와 느낌이 맞는가?")

# ---- 입력 파일 준비 : 드라이브에서 자동으로 가져온다 -------------------------
# VIII-2·VIII-3 은 입력이 **독자가 올리는 파일**이라 레인 B 의 Gradio 대역이
# `value=` 에서 꺼낼 값이 없다. 그래서 레인 B 에서 제외됐다.
# 레인 C 에는 그 파일이 드라이브에 있으므로 여기서 직접 가져와 `fn` 을 부른다.
#
# ⚠ 손으로 올리게 두지 않는다. 분기마다 반복되는 준비 작업은 **잊히거나 틀리게 되고**,
#   파일이 없으면 그 항목이 조용히 '미확인'으로 남는다.
#   폴더 경로는 **박아 두지 않고 찾아낸다** — 드라이브 구조가 바뀌어도 견디도록.

WANT = [
    "[DALL·E] gift box.png",
    "[TED-Ed] Sample Video_30s.mp4",
    "[cosmoswag_kr] Sample Video_30s.mp4",
]

def fetch_inputs():
    """드라이브를 붙여 필요한 파일만 /content/ 로 복사한다. 찾은 것 목록을 반환."""
    import shutil, unicodedata

    if not os.path.isdir("/content/drive"):
        try:
            from google.colab import drive
            drive.mount("/content/drive")       # 권한 창이 한 번 뜬다
        except Exception as e:
            print(f"  ⚠ 드라이브를 붙이지 못했다: {type(e).__name__}: {e}")
            return {}

    root = "/content/drive/MyDrive"
    if not os.path.isdir(root):
        print(f"  ⚠ {root} 가 없다")
        return {}

    # 이름 비교는 **유니코드 정규화 후**에 한다.
    # 맥과 윈도가 한글·가운뎃점을 다른 방식으로 저장해 눈으로 같아도 문자열이 다르다.
    # `[DALL·E]` 의 `·` 가 정확히 그 함정이다.
    def key(name):
        return unicodedata.normalize("NFC", name).strip().lower()

    want = {key(w): w for w in WANT}
    found = {}
    t0 = time.time()
    for dirpath, _dirnames, filenames in os.walk(root):
        for fn in filenames:
            k = key(fn)
            if k in want and k not in found:
                src = os.path.join(dirpath, fn)
                dst = "/content/" + want[k]      # 노트북이 기대하는 이름으로 맞춰 둔다
                shutil.copy(src, dst)
                found[k] = dst
                print(f"  가져옴: {fn}\n          ← {dirpath}")
        if len(found) == len(want):
            break
    print(f"  {len(found)}/{len(WANT)}개 · {time.time()-t0:.0f}초")
    for k, w in want.items():
        if k not in found:
            print(f"  🔴 못 찾음: {w}")
    return found

print("\n── 입력 파일 준비 ──")
_missing = [w for w in WANT if not os.path.exists("/content/" + w)]
if _missing:
    fetch_inputs()
else:
    print("  이미 /content/ 에 셋 다 있다 — 드라이브를 붙이지 않는다")

# ---- VIII-2 Image Captioning ------------------------------------------------
IMG = "/content/[DALL·E] gift box.png"
g_cap = run_notebook("VIII-2", "Multimodal AI_Image Captioning.ipynb")
# 셀이 실패했으면 fn 이 없다. KeyError 로 이 셀 전체를 죽이지 않는다 — 뒤의 VIII-3 도 돌아야 한다.
fn_cap = g_cap.get("generate_caption") or (_INTERFACES[-1].fn if _INTERFACES else None)
if fn_cap is None:
    print("  ⚠ generate_caption 을 못 찾았다 — 셀 0 이 끝까지 안 돌았다")
    RESULTS["VIII-2"]["err"] = "generate_caption 없음"
    RESULTS["VIII-2"]["ok"] = False
elif os.path.exists(IMG):
    from PIL import Image
    from IPython.display import display
    im = Image.open(IMG)
    display(im)
    t = time.time()
    cap = fn_cap(im)
    print(f"  캡션({time.time()-t:.0f}초): {cap}")
    RESULTS["VIII-2"]["caption"] = cap
    print("  □ 캡션이 그림 내용과 맞는가? (선물상자 / gift box)")
else:
    print(f"  ⚠ 이미지가 없다: {IMG}")
    print("     드라이브에서 못 찾았다. 파일 이름이 바뀌었는지 확인할 것")
    print("     (`·` 는 가운뎃점이다. 마침표가 아니다)")
    RESULTS["VIII-2"]["err"] = "입력 이미지 없음 — 추론 미확인"
    RESULTS["VIII-2"]["ok"] = False

# ---- VIII-3 Video Subtitle --------------------------------------------------
# 원고 점검 항목: Gradio 제목이 '영상 자막 생성 AI' 인가 (저장소는 이미 맞다)
VID_EN = "/content/[TED-Ed] Sample Video_30s.mp4"
VID_KR = "/content/[cosmoswag_kr] Sample Video_30s.mp4"
g_sub = run_notebook("VIII-3", "Multimodal AI_Video Subtitle.ipynb")
fn_sub = g_sub.get("transcribe") or (_INTERFACES[-1].fn if _INTERFACES else None)
subs = {}
if fn_sub is None:
    print("  ⚠ transcribe 를 못 찾았다 — 셀 0 이 끝까지 안 돌았다")
    RESULTS["VIII-3"]["err"] = "transcribe 없음"
    RESULTS["VIII-3"]["ok"] = False
for label, path in ([] if fn_sub is None else [("영어", VID_EN), ("한국어", VID_KR)]):
    if os.path.exists(path):
        t = time.time()
        try:
            txt = fn_sub(path)
            subs[label] = txt
            print(f"  [{label}] ({time.time()-t:.0f}초) {txt[:200]}")
        except Exception as e:
            print(f"  [{label}] 실패 — {type(e).__name__}: {e}")
    else:
        print(f"  ⚠ 영상이 없다: {path}")
RESULTS["VIII-3"]["subs"] = subs
if subs:
    print("  □ 영어 자막이 실제 발화와 맞는가?")
    print("  □ 한국어 자막이 말이 되는가? (whisper-small 은 한국어가 약하다 — 원고 설명과 대조)")
elif fn_sub is not None:
    RESULTS["VIII-3"]["err"] = "입력 영상 없음 — 추론 미확인"
    RESULTS["VIII-3"]["ok"] = False

# 여기까지의 중간 표
print("\n" + "─"*70)
for k, v in RESULTS.items():
    print(f"  {k:<8} {'완주' if v['ok'] else '실패'}  {v['secs']:>6.0f}초  {v['file']}")

In [ ]:
# ── 셀 4 · 부분 C · VI-4 · VI-5 · VI-9 번역 ──────────────────────────────────
# **가장 무겁다.** cats_vs_dogs 787MB + GloVe 822MB + IMDB 80MB + mBART 2.4GB.
# 끊기면 이 셀만 다시 누르면 된다(앞의 결과는 이미 위에 남아 있다).

# ---- VI-4 : 전이학습 vs 처음부터 -------------------------------------------
# 원고가 "Colab GPU로도 약 10분"이라 적었다. 그 문장이 맞는지 재는 것도 이 레인의 일이다.
g4 = run_notebook("VI-4", "NLP_Transfer Learning_01.ipynb")
hA, hB = g4.get("history_A"), g4.get("history_B")
if hA and hB:
    a = hA.history["val_accuracy"][-1]
    b = hB.history["val_accuracy"][-1]
    RESULTS["VI-4"]["metric"] = f"A(전이) {a:.4f} vs B(처음부터) {b:.4f}"
    print(f"\n  최종 val_accuracy — A(전이학습) {a:.4f} · B(처음부터) {b:.4f}")
    print(f"  □ 원고의 주장대로 A > B 인가?  →  {'예' if a > b else '🔴 아니다'}")
    print("  □ 사진 3장에 Actual / Model A / Model B 가 함께 찍혔는가")
    print("  □ 두 번째 그래프에서 A 곡선이 B 위에 있는가")

# ---- VI-5 : GloVe 고정 vs 미세조정 -----------------------------------------
# ⚠ 셀 6 이 input() 무한 루프다. 답은 **노트북 저장 출력에서 그대로 꺼냈다.**
#   저자가 실제로 넣었던 문장이라 책의 결과와 대조할 수 있다.
VI5_ANSWERS = [
    "I don't know how I feel about this movie.",
    "The plot was okay, but the dialogue was terrible.",
    "Great acting but the story was hard to follow.",
    "Absolutely fantastic! I loved every moment.",
    "q",
]
g5 = run_notebook("VI-5", "NLP_Transfer Learning_02.ipynb", answers=VI5_ANSWERS)
hf, ht = g5.get("history_glove_frozen"), g5.get("history_glove_finetune")
if hf and ht:
    f_, t_ = hf.history["val_accuracy"][-1], ht.history["val_accuracy"][-1]
    RESULTS["VI-5"]["metric"] = f"frozen {f_:.4f} vs finetune {t_:.4f}"
    print(f"\n  최종 val_accuracy — frozen {f_:.4f} · fine-tune {t_:.4f}")
print("  □ 네 문장의 Positive/Negative 판정이 저장 출력과 같은가")
print("     저장 출력: 0.7502/0.7145 · 0.1576/0.0496 · 0.8849/0.8808 · 0.9866/0.9933")
print("     (확률값은 학습마다 흔들린다. **판정(긍/부정)이 같은지**를 본다)")
print("  □ `input_length is deprecated` 경고가 떴는가 — 무해하다. 원고에 안내가 필요한지 판단할 것")
print("  □ 셀 6 첫 줄 주석이 '세 모델'인데 실제로는 두 개다 — 원고도 그런지 확인할 것")

# ---- VI-9 번역 : mBART-large-50 약 2.4GB -----------------------------------
g9 = run_notebook("VI-9(번역)", "NLP_Transformer-based translation model.ipynb",
                  answers=["나는 학교에 간다.", "quit"])
print("  □ 번역이 저장 출력과 같은가 — 저장 출력: 'I go to school.'")
print("  □ `convert_tokens_to_ids` 로 바꾼 코드가 실제로 동작했는가 (예외 없이 번역이 나왔는가)")

In [ ]:
# ── 셀 5 · 판정표 두 칸 ──────────────────────────────────────────────────────
# 아래 출력 전체를 복사해 지침서에 붙여넣는다.

import datetime, torch

ORDER = ["VII-2", "VII-3①", "VIII-2", "VIII-3", "VI-4", "VI-5", "VI-9(번역)"]

print("=" * 78)
print(f"레인 C-1 · 제외 7건 실행   {datetime.date.today()}   "
      f"GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음'}")
print("=" * 78)
print()
print("[코드가 판정한 것]")
print()
done = [s for s in ORDER if s in RESULTS]
okn = sum(1 for s in done if RESULTS[s]["ok"])
print(f"  돌린 것 {len(done)}/7 · 완주 {okn} · 실패 {len(done)-okn}")
print()
print(f"  {'절':<10}{'결과':<7}{'소요':>8}   노트북")
print("  " + "-" * 70)
for s in ORDER:
    if s not in RESULTS:
        print(f"  {s:<10}{'안 돌림':<7}{'':>8}")
        continue
    v = RESULTS[s]
    print(f"  {s:<10}{('완주' if v['ok'] else '실패'):<7}{v['secs']:>7.0f}초   {v['file']}")
    if v.get("metric"):
        print(f"  {'':<10}{v['metric']}")
    if v.get("caption"):
        print(f"  {'':<10}캡션: {v['caption']}")
    for lab, txt in (v.get("subs") or {}).items():
        print(f"  {'':<10}자막[{lab}]: {txt[:90]}")
    if not v["ok"] and v.get("err"):
        print(f"  {'':<10}🔴 {v['err'].splitlines()[-1][:90]}")
print()
print("  ── 셀별 소요 시간 (독자가 기다리는 시간) ──")
for s in ORDER:
    if s not in RESULTS:
        continue
    cells = " · ".join(f"{i}:{sec:.0f}초{'' if st=='OK' else '('+st+')'}"
                       for i, sec, st in RESULTS[s]["cells"])
    print(f"    {s:<10}{cells}")
print()
print("  ── 원고의 시간 문구와 대조 ──")
# ⚠ 완주한 것만 견준다. 중간에 죽은 실행의 시간을 원고와 나란히 놓으면
#    '원고가 틀렸다'로 읽히는 표가 만들어진다. 실제로 한 번 그렇게 찍었다
#    (GloVe 를 받다 만 411초를 「20분 이상」과 견줘 6.9분이라 적었다).
for s, claim in [("VI-4", "Colab GPU로도 약 10분"), ("VI-5", "20분 이상")]:
    if s not in RESULTS:
        print(f"    {s}: 원고 「{claim}」  ←→  안 돌림")
    elif not RESULTS[s]["ok"]:
        print(f"    {s}: 원고 「{claim}」  ←→  🔴 중간에 실패 — 견줄 수 없다")
    else:
        print(f"    {s}: 원고 「{claim}」  ←→  실측 {RESULTS[s]['secs']/60:.1f}분")
print()
print("-" * 78)
print()
print("[저자가 판단할 것 — 답을 적고 지침서에 함께 붙여넣는다]")
print()
for q, why in [
    ("VII-2 그림이 프롬프트(우주복 입은 개)와 맞는가",
     "씨앗이 없어 매번 다르다. 같은 그림이 아니라 '맞는 그림'인지를 본다"),
    ("VII-3① 소리가 나고 K-pop 느낌인가", "재생 버튼을 실제로 눌러 볼 것"),
    ("VIII-2 캡션이 그림 내용과 맞는가", "레인 B 는 이 추론을 한 번도 못 돌렸다"),
    ("VIII-3 한국어 자막이 쓸 만한가",
     "whisper-small 은 한국어가 약하다. 원고가 그렇게 설명하고 있는지 대조할 것"),
    ("VI-4 에서 A(전이) > B(처음부터) 가 실제로 성립했는가",
     "이 절의 논지 전체가 이 부등호 하나에 걸려 있다"),
    ("VI-5 네 문장의 긍/부정 판정이 저장 출력과 같은가",
     "확률값은 흔들린다. 판정이 뒤집혔는지만 본다"),
    ("VI-9 번역이 'I go to school.' 과 같은가", "달라도 말이 되면 통과. 원고 문구와 대조할 것"),
    ("원고의 시간 문구(약 10분 · 20분 이상)를 고쳐야 하는가",
     "무료 등급 T4 기준이다. 크게 다르면 원고를 고친다"),
]:
    print(f"  □ {q}")
    print(f"      → {why}")
    print("      답: ")
    print()
print("=" * 78)